# Bose–Hubbard Mott phase — validating the PBC (box) sampler + fixed encoding

This notebook validates that continuum VMC reproduces the **Mott-insulator** ground state of
`N` bosons in a 1D optical lattice, once two things are in place:

1. the **corrected periodic encoding** (`PeriodicBoundary.encode` now groups each particle's own
   `(sin, cos)` features — the old version scrambled them for `DeepSet`), and
2. the **box (PBC) sampler** (`SamplingConfig(box_L=L)`): MH proposals are wrapped into `[0, L)`,
   confining walkers to one unit cell and folding the spread initialisation across all wells.

**Reference:** the *continuum* N-body diagonalization `diagonalize_continuum_nbody` (the exact
ground state of the same Hamiltonian VMC solves). We compare `E_VMC` to it **after converging the
ED in grid size** — finite-difference ED converges slowly, so an under-resolved grid gives a
spuriously low reference. We also check the Mott fingerprint directly: on-site occupation
`⟨n_j⟩ → 1` and small number variance `Var(n_j)`.

See `comments/MOTT_PHASE_INVESTIGATION.md` for the full diagnosis. Parallel tempering (`sampler="pt"`)
is shown at the end as a robustness option — here it matches plain MH, but it is the right tool for
deeper lattices / larger systems where local moves can no longer cover the configuration space.

In [ ]:
import json
import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt

from qvarnet.boundaries import PeriodicBoundary
from qvarnet.models.compose import LogWavefunction
from qvarnet.models.deep_set import DeepSet
from qvarnet.hamiltonian.periodic import LatticeBoseHamiltonian
from qvarnet.config.training_setup import TrainingConfig, SamplingConfig
from qvarnet.config.coord_mode import LabCoords
from qvarnet.train import train
from qvarnet.samplers import sample_and_process, sample_parallel_tempering, geometric_betas
from qvarnet.vmc.probability import build_prob_fn

## 0. Parameters (unit-filling Mott point)

In [ ]:
N = N_SITES = 3          # bosons; unit filling N = N_SITES (one per well in the Mott limit)
A = 1.0                  # lattice spacing
L = N_SITES * A          # box length (PBC)
E_R = np.pi**2 / (2 * A**2)   # recoil energy
V0_ER = 5.0              # lattice depth in units of E_R (deep enough for a Mott well structure,
                         # shallow enough that the finite-difference ED converges at a usable grid)
V0 = V0_ER * E_R
G_1D = 3.0               # strong contact interaction -> Mott regime (U/t well above the 1D ~3.4)
SIGMA = 0.3             # Gaussian width approximating the delta interaction

print(f"N={N} bosons on {N_SITES} sites, L={L}, a={A}")
print(f"V0 = {V0:.2f}  ({V0_ER} E_R),  g_1D = {G_1D},  sigma = {SIGMA}")

## 1. Exact reference: continuum N-body diagonalization

`diagonalize_continuum_nbody` builds and diagonalizes the full continuum Hamiltonian
`H = -½ Σ ∂² + V0 Σ sin²(πx/a) + (g/σ√2π) Σ_{i<j} exp(-r²/2σ²)` on a periodic grid. It is the
exact ground state of the *same* Hamiltonian VMC solves. (As shown separately, this continuum
energy matches the Bose–Hubbard treatment to high accuracy, so it is the reference we use.)

⚠️ Memory grows as `N_grid^N`, and the routine densifies the interaction — keep `N_grid` modest
for `N≥3`.

In [ ]:
import numpy as np
from scipy.sparse import diags, kron, eye, csr_matrix
from scipy.sparse.linalg import eigsh
from itertools import combinations


def diagonalize_continuum_nbody(V0, g_1D, a, L, N_grid, N_particles, sigma,
                                dtype=np.float32):
    """
    Diagonalize the N-body continuum Hamiltonian on [0,L) with PBC.

    H = -1/2 Σ_i ∂²_xi + V0 Σ_i sin²(πx_i/a) + g_1D Σ_{i<j} V_gauss(x_i-x_j)

    Hilbert space dimension: N_grid^N_particles
    Practical limits: N<=3 with N_grid<=30, or N=4 with N_grid<=10.
    """
    dx  = dtype(L / N_grid)
    x   = np.linspace(0, L, N_grid, endpoint=False).astype(dtype)
    dim = N_grid ** N_particles

    # ── 1D single-particle operator: h = -1/2 ∂² + V(x) ──────────────────
    diag_main = np.full(N_grid,     -2.0, dtype=dtype)
    diag_off  = np.ones(N_grid - 1,       dtype=dtype)
    T1d = diags([diag_off, diag_main, diag_off], [-1, 0, 1],
                shape=(N_grid, N_grid), format='lil', dtype=dtype)
    T1d[0, -1] = dtype(1.0)   # PBC
    T1d[-1, 0] = dtype(1.0)   # PBC
    T1d = T1d.tocsr().astype(dtype) * dtype(-0.5 / dx**2)

    V1d = diags((V0 * np.sin(np.pi * x / a)**2).astype(dtype), dtype=dtype)
    h   = (T1d + V1d).astype(dtype)

    # ── One-body terms: Σ_i h_i = Σ_i I⊗...⊗h⊗...⊗I ─────────────────────
    def embed_1b(op, p):
        """Embed 1-body operator op at particle slot p."""
        left  = eye(N_grid ** p,                      format='csr', dtype=dtype)
        right = eye(N_grid ** (N_particles - p - 1),  format='csr', dtype=dtype)
        return kron(kron(left, op.astype(dtype)), right, format='csr')

    H = csr_matrix((dim, dim), dtype=dtype)
    for p in range(N_particles):
        H = (H + embed_1b(h, p)).astype(dtype)

    # ── Two-body terms: Σ_{i<j} V_gauss(x_i - x_j) ───────────────────────
    def embed_interaction_gaussian(i, j):
        """
        Gaussian-smeared contact interaction between particles i and j.
        V(x_i, x_j) = g_1D/(σ√2π) * exp(-(x_i-x_j)²/2σ²)
        This is a multiplicative operator — diagonal in position space.
        """
        Xi, Xj = np.meshgrid(x, x, indexing='ij')
        V2d = (g_1D / (sigma * np.sqrt(2 * np.pi))) * \
              np.exp(-(Xi - Xj)**2 / (2 * sigma**2))
        V2d = V2d.astype(dtype)

        # V(x_i,x_j) is diagonal in the joint (x_i,x_j) basis:
        # <xi,xj|V|xi',xj'> = V2d[xi,xj] * δ_{xi,xi'} * δ_{xj,xj'}
        n_rest    = N_grid ** (N_particles - 2)
        V2d_joint = diags(V2d.ravel(), dtype=dtype)  # (N_grid², N_grid²) diagonal

        # Embed: V2d_joint on (i,j) slots, identity on rest
        V_full = kron(V2d_joint,
                      eye(n_rest, format='csr', dtype=dtype),
                      format='csr')
        # V_full has shape (N_grid² * n_rest, N_grid² * n_rest) = (dim, dim)

        # Reshape into per-particle axes so we can permute i,j to correct slots
        n_other   = N_particles - 2
        V_dense   = V_full.toarray().reshape(
            [N_grid, N_grid] + [N_grid] * n_other +   # bra axes
            [N_grid, N_grid] + [N_grid] * n_other      # ket axes
        )
        # Current axis order: bra=(i=0, j=1, other_0=2, ...), ket=(i, j, other_0, ...)
        # Target axis order: bra=(0,1,...,N-1),               ket=(0,1,...,N-1)
        other     = [k for k in range(N_particles) if k != i and k != j]
        perm      = [i, j] + other           # current slot -> target slot
        iperm     = list(np.argsort(perm))   # target slot  -> current slot
        full_perm = iperm + [p + N_particles for p in iperm]

        V_dense = V_dense.transpose(full_perm).reshape(dim, dim).astype(dtype)
        return csr_matrix(V_dense, dtype=dtype)

    for i, j in combinations(range(N_particles), 2):
        H = (H + embed_interaction_gaussian(i, j)).astype(dtype)

    # ── Diagonalize ────────────────────────────────────────────────────────
    # eigsh needs float64 internally — cast H temporarily
    E_vals, E_vecs = eigsh(H.astype(np.float64), k=min(6, dim - 1), which='SM')
    idx = np.argsort(E_vals)
    return E_vals[idx], E_vecs[:, idx], x, dx


### 1a. Converge the reference in grid size

Finite-difference ED underestimates the (dominant) kinetic zero-point energy on a coarse grid, so
`E0` rises with `N_grid`. We sweep and extrapolate in `1/N_grid²` to get the continuum reference
`E0_ref`. **Do not trust a single coarse-grid ED value.**

In [ ]:
N_grids = [12, 16, 20, 24]
E0_grid = []
for ng in N_grids:
    ev, _, _, _ = diagonalize_continuum_nbody(
        V0=V0, g_1D=G_1D, a=A, L=L, N_grid=ng, N_particles=N, sigma=SIGMA, dtype=np.float32)
    E0_grid.append(float(ev[0]))
    print(f"N_grid={ng:2d}  dx={L/ng:.3f}  E0={E0_grid[-1]:.4f}")

inv2 = np.array([1.0 / ng**2 for ng in N_grids])
slope, intercept = np.polyfit(inv2, E0_grid, 1)   # E0(N_grid) ~ E0_ref + c/N_grid^2
E0_ref = float(intercept)
print(f"\nExtrapolated continuum reference  E0_ref ≈ {E0_ref:.4f}")

plt.figure(figsize=(5, 3))
plt.plot(inv2, E0_grid, "o-", label="ED")
plt.plot([0, inv2.max()], [E0_ref, intercept + slope * inv2.max()], "k--", label="extrapolation")
plt.scatter([0], [E0_ref], c="r", zorder=5, label=f"E0_ref={E0_ref:.3f}")
plt.xlabel("1 / N_grid²"); plt.ylabel("E0"); plt.legend(); plt.title("ED grid convergence")
plt.tight_layout(); plt.show()

## 2. Ansatz and Hamiltonian

A permutation-invariant `DeepSet` log-wavefunction on the **periodic** encoding (the fixed one),
and the shipped `LatticeBoseHamiltonian` with `boundary=PeriodicBoundary(L)` (min-image
interactions).

In [ ]:
boundary = PeriodicBoundary(L=L)

def make_model():
    return LogWavefunction(
        n_particles=N, n_dim=1, transform=boundary,
        network=DeepSet(phi_hidden=[32], F_hidden=[32]),
    )

hamiltonian = LatticeBoseHamiltonian(a=A, V0=V0, g=G_1D, sigma=SIGMA, boundary=boundary)

## 3. VMC with the box (PBC) sampler

`SamplingConfig(box_L=L)` turns on the wrapping sampler. `is_update_step_size=True` keeps the
acceptance near target.

In [ ]:
N_CHAINS = 1000
N_EPOCHS = 800

result = train(
    shape=(N_CHAINS, N),
    model=make_model(),
    optimizer=optax.adam(3e-3),
    hamiltonian=hamiltonian,
    training_config=TrainingConfig(
        n_epochs=N_EPOCHS, rng_seed=0, is_update_step_size=True,
        target_acceptance=0.5, checkpoint_path="./checkpoints/bh_mott"),
    sampler_params=SamplingConfig(
        step_size=0.3, chain_length=80, thermalization_steps=20, thinning_factor=2,
        box_L=L),                       # <-- box (PBC) sampler ON
    coord_mode=LabCoords(),
)

energies = np.array([float(s.energy) for s in result.history])
E_vmc = float(np.mean(energies[-200:]))
E_err = float(np.std(energies[-200:]) / np.sqrt(200))
print(f"E_VMC = {E_vmc:.4f} ± {E_err:.4f}    (variational upper bound)")
print(f"E0_ref = {E0_ref:.4f}    ->  relative gap = {100*(E_vmc-E0_ref)/abs(E0_ref):.2f}%")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(energies, lw=0.8, label="E_VMC")
plt.axhline(E0_ref, color="k", ls="--", label=f"ED ref {E0_ref:.3f}")
plt.xlabel("epoch"); plt.ylabel("energy"); plt.legend(); plt.title("VMC energy (box sampler)")
plt.ylim(E0_ref - 1, E0_ref + 6); plt.tight_layout(); plt.show()

## 4. The Mott fingerprint — on-site occupation and number variance

The energy alone is a weak test (it is dominated by the large band zero-point energy). The
discriminating observable is the per-well occupation `⟨n_j⟩` (→ 1 in the Mott phase, one boson per
well) and the on-site number variance `Var(n_j)` (small in the Mott phase). We sample from the
trained `|ψ|²`, fold coordinates into `[0, L)`, and count particles within ±a/2 of each site (ring
distance).

In [ ]:
def occupancy(params, sampler="mh", n_chains=1500, n_steps=400, seed=9):
    pf = build_prob_fn(make_model().apply)
    x0 = jax.random.normal(jax.random.PRNGKey(7), (n_chains, N)) * 0.7
    if sampler == "pt":
        s, _, _ = sample_parallel_tempering(
            key=jax.random.PRNGKey(seed), prob_fn=pf, prob_params=params, init_positions=x0,
            step_size=0.3, n_chains=n_chains, dof=N, n_steps=n_steps, burn_in=150, thinning=2,
            betas=geometric_betas(6, 0.05), swap_every=1, box_L=L)
    else:
        s, _, _ = sample_and_process(
            key=jax.random.PRNGKey(seed), prob_fn=pf, prob_params=params, init_positions=x0,
            step_size=0.3, n_chains=n_chains, dof=N, n_steps=n_steps, burn_in=150, thinning=2,
            box_L=L)
    x = np.mod(np.asarray(s), L)                                  # (M, N) folded
    ring_d = np.stack([np.minimum(np.abs(x - j), L - np.abs(x - j)) for j in range(N_SITES)], -1)
    n_j = (ring_d < 0.5 * A).sum(axis=1)                          # (M, N_SITES) per-site counts
    return n_j.mean(0), n_j.var(0).mean(), x

n_mean, n_var, x_samp = occupancy(result.best_params())
print(f"<n_j>   = {np.round(n_mean, 3)}   (sum = {n_mean.sum():.2f}, ideal Mott [1,1,...])")
print(f"Var(n_j) (avg over sites) = {n_var:.3f}   (0 for a perfect Fock state)")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(range(N_SITES), n_mean, color="steelblue"); ax[0].axhline(1.0, color="k", ls="--")
ax[0].set_xlabel("site j"); ax[0].set_ylabel("<n_j>"); ax[0].set_title("on-site occupation")
ax[1].hist(x_samp.ravel(), bins=60, range=(0, L), color="slategray")
for j in range(N_SITES):
    ax[1].axvline(j * A, color="r", ls="--", lw=0.7)
ax[1].set_xlabel("x"); ax[1].set_ylabel("counts"); ax[1].set_title("density n(x) (folded)")
plt.tight_layout(); plt.show()

## 5. Robustness: parallel tempering (`sampler="pt"`)

Parallel tempering stacks replicas per chain at inverse temperatures `1=β₁>β₂>…` sampling
`|ψ|^{2β}` (= `exp(-βE)` with `E ≡ -2log|ψ|`), and swaps between adjacent replicas; only the cold
β=1 replica is returned. In **this** regime it matches the box MH result — once the encoding is
fixed and wrapping gives coverage, barrier crossing is not the bottleneck. PT is the insurance for
**deeper lattices, larger N, or off-unit-filling**, where local moves can no longer cover the wells.

In [ ]:
result_pt = train(
    shape=(N_CHAINS, N),
    model=make_model(),
    optimizer=optax.adam(3e-3),
    hamiltonian=hamiltonian,
    training_config=TrainingConfig(
        n_epochs=N_EPOCHS, rng_seed=0, is_update_step_size=True,
        target_acceptance=0.5, checkpoint_path="./checkpoints/bh_mott_pt"),
    sampler_params=SamplingConfig(
        step_size=0.3, chain_length=80, thermalization_steps=20, thinning_factor=2, box_L=L,
        sampler="pt", pt_n_replicas=6, pt_beta_min=0.05, swap_every=1),   # <-- parallel tempering
    coord_mode=LabCoords(),
)
e_pt = np.array([float(s.energy) for s in result_pt.history])
E_pt = float(np.mean(e_pt[-200:]))
nm_pt, nv_pt, _ = occupancy(result_pt.best_params(), sampler="pt")
print(f"box-MH : E={E_vmc:.4f}  <n_j>={np.round(n_mean,3)}  Var={n_var:.3f}")
print(f"box-PT : E={E_pt:.4f}  <n_j>={np.round(nm_pt,3)}  Var={nv_pt:.3f}")
print(f"ED ref : E0={E0_ref:.4f}")

## 6. Conclusions

- With the **fixed periodic encoding** + **box sampler**, VMC reproduces the Mott ground state:
  `E_VMC` is a tight variational upper bound on the grid-converged continuum ED, occupation is
  `⟨n_j⟩ ≈ 1` per well, and `Var(n_j)` is small.
- The continuum ED is only trustworthy **after grid convergence**; a coarse grid under-counts the
  energy and makes a correct VMC result look too high.
- The full energy is a weak diagnostic (dominated by the band zero-point); the **occupation and
  number variance** are what confirm the Mott phase.
- **Parallel tempering** reproduces the same answer here and is available (`sampler="pt"`) as the
  robust sampler for harder regimes (deeper lattices, larger systems, off unit filling).